In [0]:
%sql
CREATE OR REPLACE TABLE bronze_orders AS
SELECT * FROM VALUES
    (101, 'Ravi', 1000, 'Completed'),
    (102, 'Priya', 2000, 'Completed'),
    (103, 'Amit', -500, 'Completed'),
    (104, 'Ravi', 1500, 'Pending'),
    (102, 'Priya', 2000, 'Completed')
AS orders(order_id, customer, amount, status);

In [0]:
%sql
CREATE OR REPLACE TABLE silver_orders AS
SELECT order_id, customer, amount, status
FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY order_id
        ) AS rn
    FROM bronze_orders
)
WHERE rn = 1
  AND amount >= 0;

In [0]:
%sql
CREATE OR REPLACE VIEW gold_customer_sales AS
SELECT
    customer,
    SUM(amount) AS total_sales
FROM silver_orders
WHERE status = 'Completed'
GROUP BY customer;